# Uniform HSBM K-sweep with Zhou theta

This notebook keeps `n=10000` and `rho_n=4` fixed while varying only the number of communities `K`.

The planted uniform HSBM uses

`p_in = a_in * rho_n / n ** (m - 1)` and `p_out = b_out * rho_n / n ** (m - 1)`.

Spectral clustering uses the Zhou operator `Theta = I - Delta` and takes the eigenvectors associated with the largest eigenvalues of `Theta`, which is equivalent to taking the smallest eigenvalues of `Delta = I - Theta`.

Note: with `a_in`, `b_out`, `m`, `n`, and `rho_n` fixed, changing `K` also changes the fraction of within-community candidate hyperedges. The notebook records both empirical and expected degree diagnostics to make this visible.


In [ ]:
from pathlib import Path
import gc
import json
import math
import sys
import time
import tracemalloc
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "common.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find project root containing src/common.py")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.common import (
    generate_planted_uniform_hsbm_instance,
    normalize_rows_l2,
    zhou_normalized_laplacian,
)

EXPERIMENT_ID = "EXP-20260427-002"
EXPERIMENT_SLUG = "uniform_hsbm_K_sweep_zhou_theta"
OUTDIR = PROJECT_ROOT / "experiments" / "균일 HSBM 실험" / "results" / f"{EXPERIMENT_ID}_{EXPERIMENT_SLUG}"
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ID, EXPERIMENT_SLUG, PROJECT_ROOT, OUTDIR


## Configuration

Only `K` is varied. `n` is fixed at `10000` and `rho_n` is fixed at `4.0`.


In [ ]:
CONFIG = {
    "n": 10_000,
    "m": 3,
    "a_in": 36.0,
    "b_out": 4.0,
    "rho_n": 4.0,
    "reps": 10,
    "seed": 20260427,
    "sampling": "sparse",
    "max_enumeration": 1_500_000,
    "normalize_embedding_rows": True,
    "eigsh_tol": 1e-6,
    "kmeans_n_init": 20,
}

K_VALUES = [2, 3, 4, 5, 6, 8, 10, 12]

CONFIG, K_VALUES


## Helpers

In [ ]:
def current_rss_mb():
    try:
        import psutil

        return psutil.Process().memory_info().rss / (1024.0 ** 2)
    except Exception:
        return np.nan


def measure_call(fn):
    gc.collect()
    rss_before_mb = current_rss_mb()
    tracemalloc.start()
    cpu_start = time.process_time()
    wall_start = time.perf_counter()
    value = fn()
    wall_clock_sec = time.perf_counter() - wall_start
    cpu_time_sec = time.process_time() - cpu_start
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rss_after_mb = current_rss_mb()
    return value, {
        "cpu_time_sec": float(cpu_time_sec),
        "wall_clock_sec": float(wall_clock_sec),
        "peak_traced_memory_mb": float(peak_bytes / (1024.0 ** 2)),
        "rss_before_mb": float(rss_before_mb) if np.isfinite(rss_before_mb) else np.nan,
        "rss_after_mb": float(rss_after_mb) if np.isfinite(rss_after_mb) else np.nan,
        "rss_delta_mb": float(rss_after_mb - rss_before_mb)
        if np.isfinite(rss_before_mb) and np.isfinite(rss_after_mb)
        else np.nan,
    }


def aligned_misclassification_rate(y_true, y_pred, K):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    conf = np.zeros((K, K), dtype=int)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < K and 0 <= p < K:
            conf[t, p] += 1
    true_ids, pred_ids = linear_sum_assignment(-conf)
    pred_to_true = {int(pred): int(true) for true, pred in zip(true_ids, pred_ids)}
    y_aligned = np.array([pred_to_true.get(int(p), int(p)) for p in y_pred], dtype=int)
    return float(np.mean(y_aligned != y_true)), y_aligned, conf


def hypergraph_vertex_degree_stats(n, hyperedges):
    degrees = np.zeros(int(n), dtype=float)
    for edge in hyperedges:
        for v in edge:
            degrees[int(v)] += 1.0
    return {
        "num_isolated_nodes": int(np.sum(degrees == 0)),
        "isolated_fraction": float(np.mean(degrees == 0)) if n > 0 else 0.0,
        "hypergraph_degree_mean": float(degrees.mean()) if n > 0 else 0.0,
        "hypergraph_degree_max": float(degrees.max()) if n > 0 else 0.0,
    }


def balanced_label_sizes(n, K):
    sizes = np.full(K, n // K, dtype=int)
    sizes[: n % K] += 1
    return sizes


def expected_uniform_hsbm_stats(n, K, m, p_in, p_out):
    sizes = balanced_label_sizes(int(n), int(K))
    total = math.comb(int(n), int(m))
    within = sum(math.comb(int(size), int(m)) for size in sizes if size >= m)
    mixed = total - within
    expected_edges = within * float(p_in) + mixed * float(p_out)
    return {
        "expected_hyperedges_total": float(expected_edges),
        "expected_hyperedges_per_n": float(expected_edges / n),
        "expected_degree_mean": float(m * expected_edges / n),
        "candidate_within_fraction": float(within / total) if total > 0 else np.nan,
    }


def spectral_cluster_from_zhou_theta(theta, K, rng, normalize_rows=True, eigsh_tol=1e-6, kmeans_n_init=20):
    n = int(theta.shape[0])
    theta = ((theta + theta.T) * 0.5).tocsr()
    total_start = time.perf_counter()
    timings = {}

    t0 = time.perf_counter()
    if n <= K + 1:
        vals, vecs = np.linalg.eigh(theta.toarray())
        order = np.argsort(vals)[-K:][::-1]
        U = vecs[:, order]
    else:
        try:
            v0 = rng.normal(size=n)
            vals, vecs = spla.eigsh(theta, k=K, which="LA", tol=eigsh_tol, v0=v0)
            order = np.argsort(vals)[::-1]
            U = vecs[:, order]
        except Exception as exc:
            warnings.warn(f"eigsh failed ({exc}); falling back to dense eigh.")
            vals, vecs = np.linalg.eigh(theta.toarray())
            order = np.argsort(vals)[-K:][::-1]
            U = vecs[:, order]
    timings["eigen_decomposition_wall_sec"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    if normalize_rows:
        U = normalize_rows_l2(U)
    timings["embedding_normalize_wall_sec"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    random_state = int(rng.integers(1, 2**31 - 1))
    labels = KMeans(n_clusters=K, n_init=int(kmeans_n_init), random_state=random_state).fit_predict(U)
    timings["kmeans_wall_sec"] = time.perf_counter() - t0
    timings["spectral_clustering_wall_sec"] = time.perf_counter() - total_start
    return labels, {
        "zhou_theta_nnz": int(theta.nnz),
        **timings,
    }


def run_one_rep(K, rep, config):
    n = int(config["n"])
    K = int(K)
    m = int(config["m"])
    rho_n = float(config["rho_n"])
    seed = int(config["seed"] + 1_000_003 * K + int(rep))
    rng = np.random.default_rng(seed)
    timings = {}

    t0 = time.perf_counter()
    hyperedges, y_true, Theta_true, gen_stats = generate_planted_uniform_hsbm_instance(
        n=n,
        K=K,
        d=m,
        a_d=float(config["a_in"]),
        b_d=float(config["b_out"]),
        rho_n=rho_n,
        rng=rng,
        sampling=config["sampling"],
        max_enumeration=int(config["max_enumeration"]),
        clip=False,
    )
    p_in = float(gen_stats["p_in"])
    p_out = float(gen_stats["p_out"])
    expected_stats = expected_uniform_hsbm_stats(n=n, K=K, m=m, p_in=p_in, p_out=p_out)
    timings["generation_wall_sec"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    L = zhou_normalized_laplacian(n=n, hyperedges=hyperedges)
    theta = (sp.eye(n, format="csr", dtype=float) - L).tocsr()
    theta.eliminate_zeros()
    timings["zhou_theta_build_wall_sec"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred, spectral_stats = spectral_cluster_from_zhou_theta(
        theta=theta,
        K=K,
        rng=rng,
        normalize_rows=bool(config["normalize_embedding_rows"]),
        eigsh_tol=float(config["eigsh_tol"]),
        kmeans_n_init=int(config["kmeans_n_init"]),
    )
    timings["spectral_clustering_wall_sec"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    mis, y_aligned, conf = aligned_misclassification_rate(y_true, y_pred, K)
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    timings["metric_wall_sec"] = time.perf_counter() - t0

    record = {
        "n": n,
        "K": K,
        "rho_n": rho_n,
        "rep": int(rep),
        "seed": seed,
        "m": m,
        "num_hyperedges_total": int(len(hyperedges)),
        "misclassification_rate": mis,
        "ARI": float(ari),
        "NMI": float(nmi),
        **timings,
        **hypergraph_vertex_degree_stats(n, hyperedges),
        **expected_stats,
        **spectral_stats,
    }
    record["algorithm_total_wall_sec"] = float(
        record["generation_wall_sec"]
        + record["zhou_theta_build_wall_sec"]
        + record["eigen_decomposition_wall_sec"]
        + record["embedding_normalize_wall_sec"]
        + record["kmeans_wall_sec"]
    )
    record["p_in"] = float(p_in)
    record["p_out"] = float(p_out)
    record["sampling_mode"] = gen_stats.get("sampling_mode", "")
    return record


def run_one_rep_measured(K, rep, config):
    record, measurement = measure_call(lambda: run_one_rep(K=K, rep=rep, config=config))
    record.update(measurement)
    return record


def run_K_experiment(K, reps=None, config=None):
    if config is None:
        config = CONFIG
    if reps is None:
        reps = int(config["reps"])

    rows = []
    for rep in range(1, reps + 1):
        row = run_one_rep_measured(K=K, rep=rep, config=config)
        rows.append(row)
        print(
            f"K={K:2d} rep={rep:2d}/{reps:<2d} "
            f"edges={row['num_hyperedges_total']:8d} "
            f"degree={row['hypergraph_degree_mean']:.2f} "
            f"expected_degree={row['expected_degree_mean']:.2f} "
            f"isolated={row['isolated_fraction']:.4f} "
            f"mis={row['misclassification_rate']:.4f} "
            f"ARI={row['ARI']:.4f} NMI={row['NMI']:.4f} "
            f"wall={row['wall_clock_sec']:.3f}s "
            f"peak={row['peak_traced_memory_mb']:.1f}MB"
        )

    df = pd.DataFrame(rows)
    display(df)
    return df


def summarize_by_K(df_raw):
    return df_raw.groupby("K", as_index=False).agg(
        reps=("rep", "count"),
        hyperedges_mean=("num_hyperedges_total", "mean"),
        hyperedges_std=("num_hyperedges_total", "std"),
        degree_mean=("hypergraph_degree_mean", "mean"),
        degree_std=("hypergraph_degree_mean", "std"),
        expected_degree_mean=("expected_degree_mean", "mean"),
        candidate_within_fraction_mean=("candidate_within_fraction", "mean"),
        isolated_fraction_mean=("isolated_fraction", "mean"),
        isolated_fraction_std=("isolated_fraction", "std"),
        misclassification_mean=("misclassification_rate", "mean"),
        misclassification_std=("misclassification_rate", "std"),
        ari_mean=("ARI", "mean"),
        ari_std=("ARI", "std"),
        nmi_mean=("NMI", "mean"),
        nmi_std=("NMI", "std"),
        generation_wall_sec_mean=("generation_wall_sec", "mean"),
        zhou_theta_build_wall_sec_mean=("zhou_theta_build_wall_sec", "mean"),
        eigen_decomposition_wall_sec_mean=("eigen_decomposition_wall_sec", "mean"),
        kmeans_wall_sec_mean=("kmeans_wall_sec", "mean"),
        spectral_clustering_wall_sec_mean=("spectral_clustering_wall_sec", "mean"),
        algorithm_total_wall_sec_mean=("algorithm_total_wall_sec", "mean"),
        cpu_time_sec_mean=("cpu_time_sec", "mean"),
        wall_clock_sec_mean=("wall_clock_sec", "mean"),
        peak_traced_memory_mb_mean=("peak_traced_memory_mb", "mean"),
        rss_delta_mb_mean=("rss_delta_mb", "mean"),
    )


## Run K sweep

In [ ]:
frames = []
for K in K_VALUES:
    frames.append(run_K_experiment(K))

df_raw = pd.concat(frames, ignore_index=True)
summary = summarize_by_K(df_raw)

display(summary)


## Save results

In [ ]:
raw_path = OUTDIR / f"{EXPERIMENT_ID}_{EXPERIMENT_SLUG}_raw.csv"
summary_path = OUTDIR / f"{EXPERIMENT_ID}_{EXPERIMENT_SLUG}_summary.csv"
config_path = OUTDIR / f"{EXPERIMENT_ID}_{EXPERIMENT_SLUG}_config.json"

df_raw.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)
with config_path.open("w", encoding="utf-8") as f:
    json.dump({"CONFIG": CONFIG, "K_VALUES": K_VALUES}, f, indent=2)

raw_path, summary_path, config_path


## Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
x = summary["K"].to_numpy()

axes[0, 0].errorbar(
    x,
    summary["misclassification_mean"],
    yerr=summary["misclassification_std"].fillna(0.0),
    marker="o",
    capsize=3,
)
axes[0, 0].set_title("Misclassification rate")
axes[0, 0].set_xlabel("K")
axes[0, 0].set_ylabel("mean +/- std")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].errorbar(x, summary["ari_mean"], yerr=summary["ari_std"].fillna(0.0), marker="o", capsize=3, label="ARI")
axes[0, 1].errorbar(x, summary["nmi_mean"], yerr=summary["nmi_std"].fillna(0.0), marker="s", capsize=3, label="NMI")
axes[0, 1].set_title("Clustering agreement")
axes[0, 1].set_xlabel("K")
axes[0, 1].set_ylabel("score")
axes[0, 1].set_ylim(-0.05, 1.05)
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend()

axes[1, 0].errorbar(x, summary["degree_mean"], yerr=summary["degree_std"].fillna(0.0), marker="o", capsize=3, label="empirical degree")
axes[1, 0].plot(x, summary["expected_degree_mean"], marker="s", label="expected degree")
axes[1, 0].plot(x, summary["candidate_within_fraction_mean"], marker="^", label="within-candidate fraction")
axes[1, 0].set_title("Density diagnostics")
axes[1, 0].set_xlabel("K")
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

axes[1, 1].errorbar(x, summary["wall_clock_sec_mean"], marker="o", capsize=3, label="wall-clock")
axes[1, 1].errorbar(x, summary["peak_traced_memory_mb_mean"], marker="s", capsize=3, label="peak traced memory MB")
axes[1, 1].set_title("Runtime and memory")
axes[1, 1].set_xlabel("K")
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

fig_path = OUTDIR / f"{EXPERIMENT_ID}_{EXPERIMENT_SLUG}_summary.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()

fig_path
